In [1]:
import json
import re
from pathlib import Path

MODEL_NAME = "Mistral-Nemo-Instruct-2407"
OUTPUT_FILE = Path(f"embeddings_{MODEL_NAME}_eigenanteil_gesamt.json")


def _part_number(path: Path):
    match = re.search(r"_Teil(\d+)\.json$", path.name)
    return int(match.group(1)) if match else 10**9


def _auto_find_input_files(model_name: str):
    pattern = f"embeddings_{model_name}_Teil*_eigenanteil.json"
    candidates = list(Path(".").glob(pattern))
    unique_sorted = sorted({p.resolve() for p in candidates}, key=lambda p: _part_number(Path(p)))
    return [Path(p) for p in unique_sorted]


INPUT_FILES = _auto_find_input_files(MODEL_NAME)
print("WORKDIR:", Path.cwd())
print("Gefundene INPUT_FILES:", len(INPUT_FILES))


def _load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _is_numeric_vector(value):
    return isinstance(value, list) and len(value) > 0 and isinstance(value[0], (int, float))


def _validate_embedding_questions(name: str, data):
    if not isinstance(data, list):
        raise ValueError(f"{name}: Erwartet oberste Ebene als Liste")
    if not data:
        raise ValueError(f"{name}: Datei ist leer")

    for q_idx, question_embeddings in enumerate(data, start=1):
        if not isinstance(question_embeddings, list) or len(question_embeddings) == 0:
            raise ValueError(f"{name}: Frage {q_idx} ist keine nicht-leere Liste")

        for s_idx, sample_embedding in enumerate(question_embeddings, start=1):
            if not _is_numeric_vector(sample_embedding):
                raise ValueError(
                    f"{name}: Frage {q_idx}, Sample {s_idx} hat kein gültiges Embedding-Format"
                )


def merge_parts_by_question(input_files, output: Path):
    input_files = [Path(p) for p in input_files]
    if len(input_files) == 0:
        raise ValueError(
            "Keine Eingabedateien gefunden. Passe MODEL_NAME an oder setze INPUT_FILES manuell."
        )

    loaded_parts = []
    for path in input_files:
        data = _load_json(path)
        _validate_embedding_questions(str(path), data)
        loaded_parts.append((path, data))

    question_count = len(loaded_parts[0][1])
    for path, data in loaded_parts[1:]:
        if len(data) != question_count:
            raise ValueError(
                f"{path}: Frageanzahl {len(data)} passt nicht zu {question_count}. "
                "Für frageweises Merging müssen alle Teile gleich viele Fragen enthalten."
            )

    merged_questions = []
    for q_idx in range(question_count):
        merged_samples = []
        for _, data in loaded_parts:
            merged_samples.extend(data[q_idx])
        merged_questions.append(merged_samples)

    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", encoding="utf-8") as f:
        json.dump(merged_questions, f, ensure_ascii=False)

    print(f"OK: {len(input_files)} Dateien frageweise zusammengeführt -> {output.resolve()}")
    print(f"Fragen gesamt: {len(merged_questions)}")
    print(f"Samples in Frage 1: {len(merged_questions[0])}")
    print(f"Embedding-Dimension in Frage 1, Sample 1: {len(merged_questions[0][0])}")


print("Verwende Dateien in Reihenfolge:")
for path in INPUT_FILES:
    print(f"- {path}")

merge_parts_by_question(INPUT_FILES, OUTPUT_FILE)

WORKDIR: /home/max/Dokumente/Master_WI/3.Semester/Data Mining/replikation-data-mining/Embeddings
Gefundene INPUT_FILES: 5
Verwende Dateien in Reihenfolge:
- /home/max/Dokumente/Master_WI/3.Semester/Data Mining/replikation-data-mining/Embeddings/embeddings_Mistral-Nemo-Instruct-2407_Teil4_eigenanteil.json
- /home/max/Dokumente/Master_WI/3.Semester/Data Mining/replikation-data-mining/Embeddings/embeddings_Mistral-Nemo-Instruct-2407_Teil2_eigenanteil.json
- /home/max/Dokumente/Master_WI/3.Semester/Data Mining/replikation-data-mining/Embeddings/embeddings_Mistral-Nemo-Instruct-2407_Teil1_eigenanteil.json
- /home/max/Dokumente/Master_WI/3.Semester/Data Mining/replikation-data-mining/Embeddings/embeddings_Mistral-Nemo-Instruct-2407_Teil3_eigenanteil.json
- /home/max/Dokumente/Master_WI/3.Semester/Data Mining/replikation-data-mining/Embeddings/embeddings_Mistral-Nemo-Instruct-2407_Teil5_eigenanteil.json
OK: 5 Dateien frageweise zusammengeführt -> /home/max/Dokumente/Master_WI/3.Semester/Data 

In [2]:
# Validierung: Prüft, ob das Merge (frageweise) korrekt ist
import numpy as np


def validate_merged_file(input_files, merged_file):
    input_files = [Path(p) for p in input_files]
    merged_file = Path(merged_file)

    part_data = [_load_json(p) for p in input_files]
    merged_data = _load_json(merged_file)

    # 1) Gleiche Frageanzahl wie in den Teilen
    question_count = len(part_data[0])
    assert all(len(d) == question_count for d in part_data), "Frageanzahl in den Teilen ist nicht konsistent"
    assert len(merged_data) == question_count, (
        f"Falsche Frageanzahl im Merge: {len(merged_data)} statt {question_count}"
    )

    # 2) Erwartete Sample-Anzahl pro Frage
    expected_samples_per_question = [sum(len(d[q]) for d in part_data) for q in range(question_count)]
    actual_samples_per_question = [len(merged_data[q]) for q in range(question_count)]
    assert actual_samples_per_question == expected_samples_per_question, "Samples pro Frage stimmen nicht"

    # 3) Embedding-Dimensionen konsistent
    dim = len(merged_data[0][0])
    assert all(len(sample) == dim for question in merged_data for sample in question), "Uneinheitliche Embedding-Dimension"

    # 4) Stichprobe: Inhalte exakt gleich zur erwarteten Konkatenation
    rng = np.random.default_rng(42)
    check_indices = rng.choice(question_count, size=min(5, question_count), replace=False)
    for q in check_indices:
        expected_question = []
        for d in part_data:
            expected_question.extend(d[q])
        assert merged_data[q] == expected_question, f"Inhalt stimmt bei Frage {q+1} nicht"

    print("✅ Merge validiert")
    print(f"Fragen: {question_count}")
    print(f"Samples je Frage (Beispiel Frage 1): {actual_samples_per_question[0]}")
    print(f"Embedding-Dimension: {dim}")


validate_merged_file(INPUT_FILES, OUTPUT_FILE)

✅ Merge validiert
Fragen: 100
Samples je Frage (Beispiel Frage 1): 50
Embedding-Dimension: 1536
